In [1]:
import os
import sys
import time

# Now import the required modules
from kubernetes import client, config
from mu_bench.deployers.manager import K8sDeploymentManager

APP_DIR = "../APPS"

In [2]:
def wait_for_pods_ready(namespace, timeout=300):
    """Wait for all pods in namespace to be ready"""
    config.load_kube_config()
    k8s_core_api = client.CoreV1Api()
    
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            pods = k8s_core_api.list_namespaced_pod(namespace=namespace)
            if not pods.items:
                time.sleep(5)
                continue
                
            ready_count = 0
            for pod in pods.items:
                if pod.status.conditions:
                    for condition in pod.status.conditions:
                        if condition.type == 'Ready' and condition.status == 'True':
                            ready_count += 1
                            break
            
            if ready_count == len(pods.items):
                print(f"✅ All {ready_count} pods ready in {namespace}")
                return True
                
            print(f"⏳ Waiting... {ready_count}/{len(pods.items)} pods ready")
            time.sleep(5)
            
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(5)
    
    print(f"❌ Timeout waiting for pods in {namespace}")
    return False

def deploy_app(app_name):
    """Deploy a single app"""
    app_dir = f"{APP_DIR}/{app_name}"
    
    print(f"🚀 Deploying {app_name}...")
    mgr = K8sDeploymentManager.from_deployment_dir(app_dir)
    mgr.deploy()
    
    print(f"⏳ Waiting for pods in {mgr.namespace}...")
    wait_for_pods_ready(mgr.namespace)

def undeploy_app(app_name):
    """Undeploy a single app"""
    app_dir = f"{APP_DIR}/{app_name}"
    
    print(f"🗑️ Undeploying {app_name}...")
    mgr = K8sDeploymentManager.from_deployment_dir(app_dir)
    mgr.undeploy()

In [3]:
# undeploy_app("app-iot-gateway")
deploy_app("app-iot-gateway")

🚀 Deploying app-iot-gateway...
[INFO] Ensuring namespace 'app-iot-gateway' exists...
[INFO] Labeled namespace 'app-iot-gateway' with istio-injection=enabled.
[INFO] Removing existing directory ../APPS/app-iot-gateway/yamls before YAML generation...
[INFO] Successfully removed ../APPS/app-iot-gateway/yamls.
[INFO] Ensured YAML directory ../APPS/app-iot-gateway/yamls exists.
[INFO] Generating YAMLs in '../APPS/app-iot-gateway/yamls'...
[INFO] Deploying resources from '../APPS/app-iot-gateway/yamls'...
[INFO] Deployment completed successfully.
⏳ Waiting for pods in app-iot-gateway...
⏳ Waiting... 0/4 pods ready
⏳ Waiting... 0/5 pods ready
⏳ Waiting... 0/5 pods ready
⏳ Waiting... 0/9 pods ready
⏳ Waiting... 0/9 pods ready
⏳ Waiting... 0/9 pods ready
⏳ Waiting... 0/9 pods ready
⏳ Waiting... 0/9 pods ready
⏳ Waiting... 0/9 pods ready
⏳ Waiting... 0/9 pods ready
⏳ Waiting... 0/9 pods ready
⏳ Waiting... 0/9 pods ready
⏳ Waiting... 1/9 pods ready
⏳ Waiting... 1/9 pods ready
⏳ Waiting... 1/9 pod